<a href="https://colab.research.google.com/github/Qaiserfarooq285/Labtasks/blob/main/Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline

In [ ]:
dataset=pd.read_csv('heart_2020_cleaned.csv')

In [ ]:
dataset.head()

In [ ]:
dataset.columns

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset= dataset.drop(['Race','SkinCancer'], axis=1)

In [ ]:
dataset.head()

In [ ]:
# Define the mapping for 'GenHealth'
genhealth_mapping = {'Poor': 0,'Fair': 1, 'Good': 2, 'Very good': 3, 'Excellent': 4}

# Define the mapping for 'AgeCategory'
age_category_mapping = {
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12
}

# Custom transformer for 'GenHealth'
def encode_genhealth(column):
    return column.replace(genhealth_mapping)

# Custom transformer for 'AgeCategory'
def encode_age_category(column):
    return column.replace(age_category_mapping)

# List of binary categorical features to encode
binary_features = ['HeartDisease','PhysicalHealth','Smoking', 'Stroke', 'Sex','AlcoholDrinking' ,'Diabetic', 'PhysicalActivity', 'KidneyDisease']

# Pipeline for binary encoding
def binary_encoder(column):
    return column.replace({'Yes': 1, 'No': 0,'N': 0,'No, borderline diabetes': 0,'Yes (during pregnancy)': 1, 'Male': 1, 'Female': 0})

# Creating transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', FunctionTransformer(binary_encoder), binary_features),
        ('genhealth', FunctionTransformer(encode_genhealth), ['GenHealth']),
        ('agecategory', FunctionTransformer(encode_age_category), ['AgeCategory'])
    ],
    remainder='passthrough'  # Keep other columns as they are
)

# Apply the transformation
processed_data = preprocessor.fit_transform(dataset)

# Convert back to DataFrame with proper column names
processed_columns = binary_features + ['GenHealth', 'AgeCategory'] + list(
    set(dataset.columns) - set(binary_features + ['GenHealth', 'AgeCategory'])
)
processed_df = pd.DataFrame(processed_data, columns=processed_columns)



In [ ]:
processed_df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Define features (X) and target (y)
X = processed_df.drop(columns=['HeartDisease'])  # Use all columns except 'HeartDisease' as features
y = processed_df['HeartDisease']  # 'HeartDisease' is the target variable

# Split the Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the KNN Model
knn = KNeighborsClassifier(n_neighbors=5)  # Adjust 'n_neighbors' as needed
knn.fit(X_train, y_train)

# Make Predictions
y_pred = knn.predict(X_test)

# Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
